# Train the dialect-to-MSA rewriter: Qwen3-1.7B + LoRA (Google Colab)

This notebook builds the training data, trains a LoRA adapter on `Qwen/Qwen3-1.7B`, publishes the adapter on the Hugging Face Hub and exports two GGUF files for Ollama: the tuned model, and the untuned base model converted in exactly the same way, so that the two can be compared like for like on your own machine.

**Before you start**

1. The golden test (`data/golden_test.jsonl`) has been corrected by one member, confirmed by a second, imported with `scripts/golden_review.py` and **committed to GitHub**. This notebook stops if it is not committed, and prints the commit date as evidence.
2. The GitHub repository is public. A private repository cannot be cloned here without a token, and a token must never be written into a notebook cell.
3. Runtime > Change runtime type > **T4 GPU**.
4. You have a Hugging Face account and a token with write access.

**Status of this notebook:** it was written without access to a GPU and has not been executed end to end. The pinned versions in `requirements-train.txt` were the latest on PyPI on 2026-11-15 and were not tested together. Run it cell by cell the first time and read each output. No number produced here is a project result; results come from `python -m src.evaluate` on your own machine.

## 1. Settings

In [ ]:
# Set these two values, then run the cell.
REPO_URL = ""      # your team's GitHub repository, e.g. "https://github.com/<team>/<repo>.git"
HF_REPO_ID = ""    # where the adapter is published, e.g. "<hf-user>/d2m-qwen3-1.7b-lora"

BASE_MODEL = "Qwen/Qwen3-1.7B"   # 1.7B parameters: inside the 4B limit of the project
GGUF_NAME = "d2m-qwen3-1.7b-q8_0.gguf"
BASE_GGUF_NAME = "qwen3-1.7b-base-q8_0.gguf"   # the untuned model, converted like the tuned one: the baseline

USE_QLORA = False   # True: load the base model in 4-bit (less GPU memory, slightly slower)
EPOCHS = 2
LEARNING_RATE = 1e-4
MAX_LENGTH = 768    # tokens per training example (system prompt + message + rewrite)
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LORA_R = 16
LORA_ALPHA = 32
SEED = 13
CORPUS_CSV = "data/raw/multidialectal_travel_corpus.csv"   # downloaded in step 4

# The corpus has almost no numbers, so the template pairs are what teaches the model to keep facts.
# These two values make about one training pair in five carry facts. They are a starting point, not a tuned choice.
CORPUS_LIMIT = 8000   # cleaned corpus pairs to use (0 = all of them, about 24,000)
SYNTHETIC_N = 2000    # template pairs with numbers, dates, times, codes and names

assert REPO_URL and HF_REPO_ID, "Fill in REPO_URL and HF_REPO_ID above."

## 2. GPU, code and libraries

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!git clone {REPO_URL} repo
%cd repo
!pip install -q -r requirements-train.txt

## 3. The golden test comes first

The cell stops if the golden test is not committed, if it differs from the committed version, or if it does not validate. The line it prints is the last commit that touched the file; that date must be earlier than this training run.

In [ ]:
import subprocess
import sys

log = subprocess.run(["git", "log", "-1", "--format=%h  %cI  %s", "--", "data/golden_test.jsonl"], capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain", "--", "data/golden_test.jsonl"], capture_output=True, text=True).stdout.strip()
assert log, "data/golden_test.jsonl is not committed in this repository. Commit and push it before training."
assert not dirty, "data/golden_test.jsonl differs from the committed version. Commit the change first."
print("golden test last commit:", log)
check = subprocess.run([sys.executable, "-m", "src.validate_golden", "data/golden_test.jsonl"], capture_output=True, text=True)
print(check.stdout[-3000:])
assert check.returncode == 0, "The golden test does not validate. Fix it, commit, and start again."

## 4. Build the training data

The corpus is "a Multidialectal Parallel Arabic Corpus - LLM" by Khalid Almeman (Zenodo record 17776617; the licence statements are in `docs/DATASET_CARD.md`). The first cell downloads the CSV (18.4 MB) with `scripts/download_corpus.py`, checks its size and MD5 against the values Zenodo publishes, and prints the columns and three pairs. The file has six columns, and every cell of `Saudi Dialect` starts with the label `- Saudi: `, which is removed on loading. If the download is refused, download the CSV by hand from https://zenodo.org/records/17776617, upload it to `data/raw/` and run the cell again: it then checks the uploaded file. The script was written without network access to Zenodo, so this cell is the first time it runs against the real site.

`src.data_prep` then drops repeated sentences and pairs whose MSA side does not carry the same numbers and Latin words as the dialect side, keeps `CORPUS_LIMIT` corpus pairs, adds `SYNTHETIC_N` seeded template pairs, adds any team-written files named `data/team_*.jsonl`, removes every pair that is close to a golden text or shares a distinctive fact with one, and fails if any overlap remains. Download `data/processed/prep_report.json` with the last cell of this step and commit it: it is the proof for the dataset card.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "scripts/download_corpus.py", "--out", CORPUS_CSV], capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-2000:])
assert result.returncode == 0, "the corpus was not downloaded or is not the expected file; read the message above"

In [ ]:
import glob
import subprocess
import sys

cmd = [sys.executable, "-m", "src.data_prep",
       "--golden", "data/golden_test.jsonl",
       "--corpus-csv", CORPUS_CSV,
       "--corpus-limit", str(CORPUS_LIMIT),
       "--synthetic-n", str(SYNTHETIC_N),
       "--seed", str(SEED),
       "--out", "data/processed"]
team_files = sorted(glob.glob("data/team_*.jsonl"))
if team_files:
    cmd += ["--team-jsonl", *team_files]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:], result.stderr[-2000:])
assert result.returncode == 0, "data preparation failed; read the message above"

In [ ]:
# Save the report on your computer, then commit it to the repository as data/processed/prep_report.json.
try:
    from google.colab import files
    files.download("data/processed/prep_report.json")
except ImportError:
    print("Not running in Colab: copy data/processed/prep_report.json by hand.")

## 5. Load the data and look at what the model will see

Compare the two printouts. The training example should contain the same turn markers as the inference prompt, including the empty `<think>` block that Qwen3 uses when thinking is off. If the training text has no `<think>` block while the inference prompt has one, tell the team before training: the model would be trained in a slightly different format from the one it is served in.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("json", data_files={"train": "data/processed/train.jsonl",
                                      "validation": "data/processed/val.jsonl"})
ds = ds.select_columns(["messages"])
print(ds)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
example = ds["train"][0]["messages"]
print("=== one training example as the model sees it ===")
print(tok.apply_chat_template(example, tokenize=False))
print("=== the prompt used at inference (thinking off) ===")
print(tok.apply_chat_template(example[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False))

sample = ds["train"].select(range(min(500, len(ds["train"]))))["messages"]
lengths = [len(tok(tok.apply_chat_template(m, tokenize=False))["input_ids"]) for m in sample]
too_long = sum(n > MAX_LENGTH for n in lengths) / len(lengths)
print(f"token length: max {max(lengths)}, share above MAX_LENGTH: {too_long:.1%}")

## 6. Train the LoRA adapter

`assistant_only_loss=True` puts the loss on the MSA rewrite only, not on the instruction or the dialect message. If the GPU runs out of memory, set `USE_QLORA = True` in the settings cell, or lower `BATCH_SIZE` and raise `GRAD_ACCUMULATION` by the same factor, then restart the runtime.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

if USE_QLORA:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    model_or_id = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
        device_map="auto",
    )
else:
    model_or_id = BASE_MODEL

peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")

training_args = SFTConfig(
    output_dir="outputs/d2m-qwen3-lora",
    assistant_only_loss=True,
    max_length=MAX_LENGTH,
    packing=False,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    seed=SEED,
    report_to="none",
    hub_model_id=HF_REPO_ID,
)

trainer = SFTTrainer(
    model=model_or_id,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

## 7. Save the adapter and confirm it is an adapter

In [ ]:
import os

ADAPTER_DIR = "outputs/d2m-qwen3-lora/final_adapter"
trainer.save_model(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
files = sorted(os.listdir(ADAPTER_DIR))
print(files)
assert "adapter_config.json" in files, "Expected a LoRA adapter (adapter_config.json), not full model weights."

## 8. Smoke check on a few validation pairs

This only shows that the tuned model answers with a rewrite and what the fact checker says about it. It is not the evaluation: the validation split comes from the training sources, not from the golden test.

In [ ]:
from src.facts import compare_facts
from src.pipeline import clean_output

model = trainer.model
model.eval()
for row in ds["validation"].select(range(min(5, len(ds["validation"])))):
    messages = row["messages"]
    prompt = tok.apply_chat_template(messages[:-1], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    reply = clean_output(tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    check = compare_facts(messages[1]["content"], reply)
    print("DIALECT :", messages[1]["content"])
    print("EXPECTED:", messages[2]["content"])
    print("MODEL   :", reply)
    print("        -> facts", "kept" if check.ok else f"DIFFER ({check.kind})", "\n")

## 9. Publish the adapter and its model card on the Hub

`login()` asks for your token. Do not paste the token into a cell and do not commit it. Fill in `docs/MODEL_CARD.md` with your real evaluation results once you have them and upload it again; until then the card says the results are pending.

In [ ]:
from huggingface_hub import HfApi, login

login()
api = HfApi()
api.create_repo(HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=ADAPTER_DIR, repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj="docs/MODEL_CARD.md", path_in_repo="README.md", repo_id=HF_REPO_ID)
print("published: https://huggingface.co/" + HF_REPO_ID)

## 10. Merge the adapter and export GGUF files for Ollama

Ollama runs GGUF files. The adapter is merged into a full-precision copy of the base model and converted with the script from the llama.cpp repository (`q8_0`: 8-bit weights; a ready-made `Q8_0` file of this model on the Hub is listed at 2.17 GB).

The untuned base model is saved from the same half-precision copy, before the adapter is merged into it, and converted with the same script and the same setting. This is the baseline of the evaluation. The `qwen3:1.7b` build in the Ollama library is a 4-bit file (`Q4_K_M`), so comparing it with an 8-bit tuned file would mix the effect of quantisation with the effect of fine-tuning.

The published artefact that the project requires is still the adapter from step 9; the GGUF files are only how the two models are served locally.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer, model
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)
# The untuned model is saved first, from the same half-precision copy, so that the baseline goes through the same steps.
BASE_DIR = "outputs/base"
base.save_pretrained(BASE_DIR)
tok.save_pretrained(BASE_DIR)

# Merging changes `base` in place, which is why the untuned copy is saved before this line.
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
MERGED_DIR = "outputs/merged"
merged.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
print(sorted(os.listdir(MERGED_DIR)))

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q sentencepiece protobuf
!python /content/llama.cpp/convert_hf_to_gguf.py outputs/merged --outfile outputs/{GGUF_NAME} --outtype q8_0
!ls -lh outputs/{GGUF_NAME}

In [ ]:
# The untuned base model saved in the cell above, converted with the same script and the same setting.
!python /content/llama.cpp/convert_hf_to_gguf.py {BASE_DIR} --outfile outputs/{BASE_GGUF_NAME} --outtype q8_0
!ls -lh outputs/{BASE_GGUF_NAME}

In [ ]:
# Store both GGUF files next to the adapter on the Hub so every team member can download them.
for name in (GGUF_NAME, BASE_GGUF_NAME):
    api.upload_file(path_or_fileobj=f"outputs/{name}", path_in_repo=f"gguf/{name}", repo_id=HF_REPO_ID)
print(f"download from: https://huggingface.co/{HF_REPO_ID}/tree/main/gguf")

## 11. Record the exact library versions

Commit `requirements-train.lock.txt` to the repository. It lists the versions that actually ran, which is what "pinned versions" has to mean for the training code.

In [ ]:
!pip freeze > requirements-train.lock.txt
!wc -l requirements-train.lock.txt

## 12. Back on your own machine

```bash
# 1. Download the two files in gguf/ from the Hub repository's Files tab into models/
# 2. Create the two Ollama models (both reuse the chat template of qwen3:1.7b, which is pulled only for that)
ollama pull qwen3:1.7b
python scripts/make_modelfile.py --gguf models/qwen3-1.7b-base-q8_0.gguf --name d2m-qwen3-base --out Modelfile.base
ollama create d2m-qwen3-base -f Modelfile.base
python scripts/make_modelfile.py --gguf models/d2m-qwen3-1.7b-q8_0.gguf
ollama create d2m-qwen3-lora -f Modelfile

# 3. Evaluate the three systems on the golden test, then run the safety check
python -m src.evaluate --golden data/golden_test.jsonl --base-model d2m-qwen3-base --tuned-model d2m-qwen3-lora
# copy results/judgments_template.csv to results/judgments.csv; two members fill in the copy, then:
python -m src.evaluate --golden data/golden_test.jsonl --from-predictions results --judgments results/judgments.csv
python -m src.safety_check --model d2m-qwen3-lora

# 4. Demo
streamlit run app/streamlit_app.py
```